# BirdCLEF 2026 — dataset preparation
Milestone I: data validation, exploration, label encoding, spectrogram export,
recording-level train/validation/test splits and a reusable data loader.
No neural network is created or trained.

Run cells in order with the original competition dataset attached. Section 5–6
uses the first available GPU, 64 mel bands, a 20 ms hop and full recording bags.
`PILOT_FILES_PER_SOURCE = None` exports the complete dataset; set it to 200 for
a trial. Each run creates a new export directory and does not resume a prior run.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import random
import sys
from pathlib import Path

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ast
import soundfile as sf

from IPython.display import display

import tensorflow as tf

print(f"Python version: {sys.version.split()[0]}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"Available GPUs: {tf.config.list_physical_devices('GPU')}")

# ============================================================
# CONFIGURATION
# ============================================================

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Dataset split ratios
AUDIO_TRAIN_RATIO = 0.80
AUDIO_VAL_RATIO = 0.10
AUDIO_TEST_RATIO = 0.10

SOUNDSCAPE_TRAIN_RATIO = 0.70
SOUNDSCAPE_VAL_RATIO = 0.15
SOUNDSCAPE_TEST_RATIO = 0.15

# Audio preprocessing
SAMPLE_RATE = 32000
WINDOW_SECONDS = 5.0
N_MELS = 64

# Audio sampling
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_SECONDS)

# Execution settings
QUICK_TEST = False
MAX_TEST_FILES = 200

# Validate configuration
assert np.isclose(
    AUDIO_TRAIN_RATIO + AUDIO_VAL_RATIO + AUDIO_TEST_RATIO, 1.0
), "Audio split ratios must sum to 1."

assert np.isclose(
    SOUNDSCAPE_TRAIN_RATIO + SOUNDSCAPE_VAL_RATIO
    + SOUNDSCAPE_TEST_RATIO, 1.0
), "Soundscape split ratios must sum to 1."

print("Configuration initialized successfully.")
print(f"Samples per window: {WINDOW_SAMPLES:,}")

# Full-recording MIL: no crop and no maximum bag size.
FFT_LENGTH = 2048
FRAME_LENGTH = 2048
HOP_LENGTH = 640
MEL_LOWER_HZ = 20.0
MEL_UPPER_HZ = 14000.0
LOG_EPSILON = 1e-6
BATCH_SIZE = 1  # Full recordings can be long; raise only if memory allows.
PREFETCH_BATCHES = 1

DETERMINISTIC = True
tf.keras.utils.set_random_seed(SEED)
if DETERMINISTIC:
    tf.config.experimental.enable_op_determinism()
    
assert 0 <= MEL_LOWER_HZ < MEL_UPPER_HZ <= SAMPLE_RATE / 2
assert FRAME_LENGTH <= FFT_LENGTH and WINDOW_SAMPLES >= FRAME_LENGTH
assert BATCH_SIZE >= 1

In [ ]:
# ============================================================
# DATASET DISCOVERY — DIRECT PATH
# ============================================================

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

DATA_DIR = Path("/kaggle/input/competitions/birdclef-2026")

if not (DATA_DIR / "train.csv").is_file():
    raise FileNotFoundError(
        f"Nem található train.csv ebben a mappában: {DATA_DIR}\n"
        "A DATA_DIR értékét a Kaggle Input panel alapján állítsd be."
    )

print(f"Dataset directory: {DATA_DIR}")

# ============================================================
# BASIC VALIDATION
# ============================================================

required_files = [
    "train.csv",
    "taxonomy.csv",
    "train_soundscapes_labels.csv",
]

required_directories = [
    "train_audio",
    "train_soundscapes",
]

missing = [
    str(DATA_DIR / filename)
    for filename in required_files
    if not (DATA_DIR / filename).is_file()
]

missing.extend(
    str(DATA_DIR / dirname)
    for dirname in required_directories
    if not (DATA_DIR / dirname).is_dir()
)

if missing:
    raise FileNotFoundError(
        "Missing dataset components:\n" + "\n".join(missing)
    )
    
print("\nAll required dataset components are available.")

# ============================================================
# 2.1 - LOAD DATASETS AND VALIDATE REQUIRED COLUMNS
# ============================================================

import ast
import soundfile as sf

train_df = pd.read_csv(
    DATA_DIR / "train.csv",
    dtype={
        "filename": "string",
        "primary_label": "string",
        "secondary_labels": "string",
    },
)

taxonomy_df = pd.read_csv(
    DATA_DIR / "taxonomy.csv",
    dtype={"primary_label": "string"},
)

soundscape_df = pd.read_csv(
    DATA_DIR / "train_soundscapes_labels.csv",
    dtype={
        "filename": "string",
        "start": "string",
        "end": "string",
        "primary_label": "string",
    },
)

required_columns = {
    "train": {"filename", "primary_label", "secondary_labels"},
    "taxonomy": {"primary_label"},
    "soundscape": {"filename", "start", "end", "primary_label"},
}

tables = {
    "train": train_df,
    "taxonomy": taxonomy_df,
    "soundscape": soundscape_df,
}

for name, frame in tables.items():
    missing = required_columns[name] - set(frame.columns)

    if missing:
        raise ValueError(
            f"{name}: missing required columns: {sorted(missing)}"
        )

    if frame.empty:
        raise ValueError(f"{name}: the table is empty.")

print("2.1 OK: datasets loaded and required columns verified.")


# ============================================================
# 2.2 - VALIDATE REQUIRED VALUES AND CLASS LABELS
# ============================================================

def require_nonempty(frame, columns, name):
    """Reject missing or whitespace-only required values."""
    for column in columns:
        values = frame[column].astype("string").str.strip()
        invalid = values.isna() | values.eq("").fillna(False)

        if invalid.any():
            rows = frame.index[invalid].tolist()[:5]
            raise ValueError(
                f"{name}: empty '{column}' values at rows {rows}."
            )

        frame[column] = values


def parse_label_field(value, list_format=False):
    """Parse labels without silently discarding malformed values."""
    if pd.isna(value) or str(value).strip() == "":
        if list_format:
            return []
        raise ValueError("An annotation label field is empty.")

    text = str(value).strip()

    if list_format or text.startswith("["):
        try:
            labels = ast.literal_eval(text)
        except (ValueError, SyntaxError) as error:
            raise ValueError(
                f"Invalid label list: {text!r}"
            ) from error

        if not isinstance(labels, (list, tuple)):
            raise ValueError(f"Expected a label list: {text!r}")
    else:
        labels = text.split(";")

    labels = [str(label).strip() for label in labels]

    if any(not label for label in labels):
        raise ValueError(f"Empty label inside field: {text!r}")

    return list(dict.fromkeys(labels))


require_nonempty(train_df, ["filename", "primary_label"], "train")
require_nonempty(taxonomy_df, ["primary_label"], "taxonomy")
require_nonempty(
    soundscape_df,
    ["filename", "start", "end", "primary_label"],
    "soundscape",
)

# Normalize path separators.
for frame in [train_df, soundscape_df]:
    frame["filename"] = frame["filename"].str.replace(
        "\\", "/", regex=False
    )

known_labels = set(taxonomy_df["primary_label"])

secondary_lists = []
for row_index, value in train_df["secondary_labels"].items():
    try:
        secondary_lists.append(
            parse_label_field(value, list_format=True)
        )
    except ValueError as error:
        raise ValueError(
            f"train: invalid secondary_labels at row {row_index}: {error}"
        ) from error

soundscape_lists = []
for row_index, value in soundscape_df["primary_label"].items():
    try:
        labels = parse_label_field(value)
        if not labels:
            raise ValueError("Annotation has no class labels.")
        soundscape_lists.append(labels)
    except ValueError as error:
        raise ValueError(
            f"soundscape: invalid labels at row {row_index}: {error}"
        ) from error

used_labels = set(train_df["primary_label"])
used_labels.update(
    label for labels in secondary_lists for label in labels
)
used_labels.update(
    label for labels in soundscape_lists for label in labels
)

unknown_labels = used_labels - known_labels

if unknown_labels:
    raise ValueError(
        f"Labels missing from taxonomy: {sorted(unknown_labels)}"
    )

# Keep CSV-compatible strings for compatibility with later blocks.
train_df["secondary_labels"] = [
    repr(sorted(labels)) for labels in secondary_lists
]
soundscape_df["primary_label"] = [
    ";".join(sorted(labels)) for labels in soundscape_lists
]

print("2.2 OK: required values and class labels verified.")


# ============================================================
# 2.3 - REMOVE DUPLICATES AND VALIDATE ANNOTATION INTERVALS
# ============================================================

for name in tables:
    tables[name] = tables[name].drop_duplicates().reset_index(drop=True)

train_df = tables["train"]
taxonomy_df = tables["taxonomy"]
soundscape_df = tables["soundscape"]

if train_df["filename"].duplicated().any():
    repeated = train_df.loc[
        train_df["filename"].duplicated(keep=False), "filename"
    ].unique()[:5]

    raise ValueError(
        f"train: multiple distinct rows reference the same file: "
        f"{repeated.tolist()}"
    )

if taxonomy_df["primary_label"].duplicated().any():
    repeated = taxonomy_df.loc[
        taxonomy_df["primary_label"].duplicated(keep=False),
        "primary_label",
    ].unique()[:5]

    raise ValueError(
        f"taxonomy: conflicting rows for class identifiers: "
        f"{repeated.tolist()}"
    )


def parse_time_seconds(value):
    """Accept numeric seconds or HH:MM:SS timestamps."""
    try:
        seconds = float(value)
    except (TypeError, ValueError):
        try:
            seconds = pd.to_timedelta(str(value)).total_seconds()
        except (TypeError, ValueError, OverflowError) as error:
            raise ValueError(
                f"Invalid annotation time: {value!r}"
            ) from error

    if not np.isfinite(seconds):
        raise ValueError(f"Nonfinite annotation time: {value!r}")

    return seconds


intervals = soundscape_df.copy()
intervals["start"] = intervals["start"].map(parse_time_seconds)
intervals["end"] = intervals["end"].map(parse_time_seconds)

invalid = (
    (intervals["start"] < 0)
    | (intervals["end"] <= intervals["start"])
    | ~np.isclose(
        intervals["end"] - intervals["start"],
        WINDOW_SECONDS,
        atol=1e-6,
        rtol=0,
    )
)

if invalid.any():
    examples = intervals.loc[
        invalid, ["filename", "start", "end"]
    ].head(5)

    raise ValueError(
        f"Invalid annotation intervals:\n{examples.to_string(index=False)}"
    )


def merge_interval_labels(values):
    """Union labels only within the same recording and interval."""
    labels = {
        label
        for value in values
        for label in value.split(";")
    }
    return ";".join(sorted(labels))


soundscape_df = (
    intervals
    .groupby(["filename", "start", "end"], as_index=False, sort=True)
    .agg(primary_label=("primary_label", merge_interval_labels))
)

# Preserve timestamp strings expected by the existing exploration cells.
for column in ["start", "end"]:
    soundscape_df[column] = soundscape_df[column].map(
        lambda seconds: str(pd.to_timedelta(seconds, unit="s"))
    )

print("2.3 OK: duplicates resolved and annotation intervals verified.")


# ============================================================
# 2.4 - MATCH ANNOTATIONS TO AUDIO FILES
# ============================================================

audio_dir = DATA_DIR / "train_audio"
soundscape_dir = DATA_DIR / "train_soundscapes"

# Verify every referenced file; no recursive directory scan.
for name, frame, directory in [
    ("train", train_df, audio_dir),
    ("soundscape", soundscape_df, soundscape_dir),
]:
    missing = [
        filename
        for filename in frame["filename"].unique()
        if not (directory / filename).is_file()
    ]

    if missing:
        raise FileNotFoundError(
            f"{name}: {len(missing)} referenced files are missing. "
            f"Examples: {missing[:5]}"
        )

# Read each referenced soundscape header once.
soundscape_info = {}

for filename in soundscape_df["filename"].unique():
    try:
        info = sf.info(str(soundscape_dir / filename))
    except Exception as error:
        raise RuntimeError(
            f"Cannot read soundscape header: {filename}"
        ) from error

    if info.samplerate <= 0 or info.frames <= 0:
        raise ValueError(f"Invalid soundscape header: {filename}")

    soundscape_info[filename] = info

for row in soundscape_df.itertuples(index=False):
    info = soundscape_info[row.filename]

    start_frame = round(parse_time_seconds(row.start) * info.samplerate)
    end_frame = round(parse_time_seconds(row.end) * info.samplerate)

    if not (0 <= start_frame < end_frame <= info.frames):
        raise ValueError(
            f"Annotation outside recording: {row.filename}, "
            f"{row.start}–{row.end}"
        )

# Keep variables required by the current downstream blocks.
train_clean_df = train_df.copy()
taxonomy_clean_df = taxonomy_df.copy()
soundscape_clean_df = soundscape_df.copy()

datasets = {
    "Training metadata": train_df,
    "Taxonomy": taxonomy_df,
    "Soundscape labels": soundscape_df,
}

print("2.4 OK: referenced audio files and annotation boundaries verified.")

In [ ]:
# ============================================================
# 3.1 - DATASET OVERVIEW AND CLASS DISTRIBUTION
# ============================================================

import soundfile as sf
import librosa
import librosa.display
from IPython.display import Audio, display

overview = pd.DataFrame({
    "Data": [
        "Individual audio recordings",
        "Annotated soundscape recordings",
        "Annotated soundscape intervals",
        "Taxonomy classes",
    ],
    "Count": [
        len(train_df),
        soundscape_clean_df["filename"].nunique(),
        len(soundscape_clean_df),
        taxonomy_df["primary_label"].nunique(),
    ],
})

display(overview)

class_counts = train_df["primary_label"].value_counts()

print("Represented primary classes:", len(class_counts))
print("Minimum recordings per class:", class_counts.min())
print("Median recordings per class:", class_counts.median())
print("Maximum recordings per class:", class_counts.max())

plt.figure(figsize=(9, 5))
plt.hist(class_counts.values, bins=40, edgecolor="black")
plt.title("Distribution of Recordings per Class")
plt.xlabel("Recordings per Class")
plt.ylabel("Number of Classes")
plt.tight_layout()
plt.show()


# ============================================================
# 3.2 - RECORDING RATINGS AND SOUNDSCAPE LABEL COUNTS
# ============================================================

if "rating" in train_df.columns:
    ratings = pd.to_numeric(train_df["rating"], errors="coerce")

    plt.figure(figsize=(9, 5))
    plt.hist(ratings.dropna(), bins=20, edgecolor="black")
    plt.title("Distribution of Recording Ratings")
    plt.xlabel("Rating")
    plt.ylabel("Number of Recordings")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# Reuse the label parser defined in Section 2.
species_counts = soundscape_clean_df["primary_label"].map(
    lambda value: len(set(parse_label_field(value)))
)

count_distribution = species_counts.value_counts().sort_index()

plt.figure(figsize=(9, 5))
plt.bar(
    count_distribution.index,
    count_distribution.values,
    edgecolor="black",
)
plt.xticks(count_distribution.index)
plt.title("Species per Annotated Soundscape Interval")
plt.xlabel("Number of Annotated Species")
plt.ylabel("Number of Intervals")
plt.tight_layout()
plt.show()


# ============================================================
# 3.3 - RECORDING DURATION DISTRIBUTION
# ============================================================

# Use a reproducible sample to avoid reading every audio header.
DURATION_SAMPLE_SIZE = 200

audio_sample = train_df.sample(
    n=min(DURATION_SAMPLE_SIZE, len(train_df)),
    random_state=SEED,
).copy()


def get_audio_duration(path):
    """Read recording duration without decoding the waveform."""
    info = sf.info(str(path))
    return info.frames / info.samplerate


audio_sample["duration_seconds"] = audio_sample["filename"].map(
    lambda filename: get_audio_duration(audio_dir / filename)
)

print(f"Recordings sampled for duration analysis: {len(audio_sample)}")

plt.figure(figsize=(9, 5))
plt.hist(
    audio_sample["duration_seconds"],
    bins=30,
    edgecolor="black",
)
plt.axvline(
    WINDOW_SECONDS,
    color="red",
    linestyle="--",
    label=f"{WINDOW_SECONDS:g}-second window",
)
plt.title("Recording Duration Distribution — Sample")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Recordings")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 3.4 - EXAMPLE AUDIO, WAVEFORM AND MEL SPECTROGRAM
# ============================================================

example = audio_sample.iloc[0]
audio_path = audio_dir / example["filename"]

# Show the same excerpt in the player and both plots.
waveform, sr = librosa.load(
    audio_path,
    sr=SAMPLE_RATE,
    duration=WINDOW_SECONDS,
    mono=True,
)

print("Filename:", example["filename"])
print("Primary label:", example["primary_label"])
print(f"Excerpt duration: {len(waveform) / sr:.2f} seconds")
print(f"Sample rate: {sr:,} Hz")

display(Audio(waveform, rate=sr))

# Illustrative visualization; training features are generated in Section 7.
mel_power = librosa.feature.melspectrogram(
    y=waveform,
    sr=sr,
    n_fft=FFT_LENGTH,
    win_length=FRAME_LENGTH,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    fmin=MEL_LOWER_HZ,
    fmax=MEL_UPPER_HZ,
    power=2.0,
)

mel_db = librosa.power_to_db(mel_power, ref=np.max)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

librosa.display.waveshow(waveform, sr=sr, ax=axes[0])
axes[0].set_title("Example Audio Waveform")
axes[0].set_xlabel("Time (seconds)")
axes[0].set_ylabel("Amplitude")

image = librosa.display.specshow(
    mel_db,
    sr=sr,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="mel",
    fmin=MEL_LOWER_HZ,
    fmax=MEL_UPPER_HZ,
    ax=axes[1],
)
axes[1].set_title("Example Mel Spectrogram (dB)")
fig.colorbar(image, ax=axes[1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.1 - CLASS MAPPING AND TARGET ENCODING
# ============================================================

class_names = taxonomy_clean_df["primary_label"].tolist()
N_CLASSES = len(class_names)

class_to_idx = {
    label: index for index, label in enumerate(class_names)
}
idx_to_class = {
    index: label for index, label in enumerate(class_names)
}


def encode_labels(labels, class_to_idx, n_classes):
    """Encode labels using the shared taxonomy order."""
    vector = np.zeros(n_classes, dtype=np.uint8)

    for label in labels:
        if label not in class_to_idx:
            raise ValueError(f"Unknown class label: {label}")
        vector[class_to_idx[label]] = 1

    return vector


# Individual recordings: preserve primary and secondary roles.
train_clean_df["primary_list"] = (
    train_clean_df["primary_label"].map(lambda label: [label])
)
train_clean_df["secondary_list"] = (
    train_clean_df["secondary_labels"].map(
        lambda value: parse_label_field(value, list_format=True)
    )
)

# Soundscapes: all interval labels enter the primary vector.
soundscape_clean_df["primary_list"] = (
    soundscape_clean_df["primary_label"].map(parse_label_field)
)
soundscape_clean_df["secondary_list"] = [
    [] for _ in range(len(soundscape_clean_df))
]

# Keep label unions for the subsequent class-aware splitting.
# No combined target vector is stored.
for frame in [train_clean_df, soundscape_clean_df]:
    frame["all_labels"] = [
        list(dict.fromkeys(primary + secondary))
        for primary, secondary in zip(
            frame["primary_list"], frame["secondary_list"]
        )
    ]

audio_primary_targets = np.stack([
    encode_labels(labels, class_to_idx, N_CLASSES)
    for labels in train_clean_df["primary_list"]
])
audio_secondary_targets = np.stack([
    encode_labels(labels, class_to_idx, N_CLASSES)
    for labels in train_clean_df["secondary_list"]
])

soundscape_primary_targets = np.stack([
    encode_labels(labels, class_to_idx, N_CLASSES)
    for labels in soundscape_clean_df["primary_list"]
])
soundscape_secondary_targets = np.stack([
    encode_labels(labels, class_to_idx, N_CLASSES)
    for labels in soundscape_clean_df["secondary_list"]
])

print("OK: shared class mapping and separate target vectors created.")


# ============================================================
# 4.2 - VALIDATE TARGETS AND RECORD IDENTIFIERS
# ============================================================

def validate_targets(frame, primary, secondary, source):
    """Verify dimensions, values and exact label reconstruction."""
    expected_shape = (len(frame), N_CLASSES)

    for name, targets in [
        ("primary", primary),
        ("secondary", secondary),
    ]:
        if targets.shape != expected_shape:
            raise ValueError(
                f"{source}: invalid {name} target shape: {targets.shape}"
            )

        if targets.dtype != np.uint8 or not np.isin(targets, [0, 1]).all():
            raise ValueError(
                f"{source}: {name} targets must be binary uint8 arrays."
            )

        for position, (labels, vector) in enumerate(
            zip(frame[f"{name}_list"], targets)
        ):
            recovered = {
                idx_to_class[index] for index in np.flatnonzero(vector)
            }

            if recovered != set(labels):
                raise ValueError(
                    f"{source}: {name} label mismatch at position {position}."
                )

    if source == "audio":
        if not np.all(primary.sum(axis=1) == 1):
            raise ValueError(
                "audio: each recording must have exactly one primary label."
            )
    else:
        if not np.all(primary.sum(axis=1) >= 1):
            raise ValueError(
                "soundscape: each interval must have annotated labels."
            )

        if secondary.any():
            raise ValueError(
                "soundscape: secondary targets must contain only zeros."
            )


if "rating" not in train_clean_df.columns:
    raise ValueError("train.csv must contain the rating column to preserve ratings.")
audio_metadata = train_clean_df[["filename"]].copy()
audio_metadata["rating"] = pd.to_numeric(
    train_clean_df["rating"], errors="raise"
).to_numpy(dtype=np.float64)
audio_metadata["source"] = "audio"
audio_metadata["sample_id"] = (
    "audio:" + audio_metadata["filename"]
)

soundscape_metadata = soundscape_clean_df[
    ["filename", "start", "end"]
].copy()
soundscape_metadata["source"] = "soundscape"
soundscape_metadata["rating"] = np.nan  # No recording rating is supplied.

for column in ["start", "end"]:
    soundscape_metadata[column] = (
        soundscape_metadata[column].map(parse_time_seconds)
    )

soundscape_metadata["sample_id"] = [
    "soundscape:" + json.dumps(
        [filename, start, end], ensure_ascii=False
    )
    for filename, start, end in zip(
        soundscape_metadata["filename"],
        soundscape_metadata["start"],
        soundscape_metadata["end"],
    )
]

target_sets = {
    "audio": (
        train_clean_df,
        audio_metadata,
        audio_primary_targets,
        audio_secondary_targets,
    ),
    "soundscape": (
        soundscape_clean_df,
        soundscape_metadata,
        soundscape_primary_targets,
        soundscape_secondary_targets,
    ),
}

for source, (frame, metadata, primary, secondary) in target_sets.items():
    validate_targets(frame, primary, secondary, source)

    if len(metadata) != len(frame) or not metadata["sample_id"].is_unique:
        raise ValueError(
            f"{source}: missing or duplicated sample identifiers."
        )

print("OK: target vectors and sample identifiers verified.")


# ============================================================
# 4.3 - EXPORT AND VERIFY SAVED LABEL DATA
# ============================================================

LABEL_DIR = OUTPUT_DIR / "labels"
LABEL_DIR.mkdir(parents=True, exist_ok=True)

class_mapping_df = pd.DataFrame({
    "class_index": np.arange(N_CLASSES),
    "class_label": class_names,
})

mapping_path = LABEL_DIR / "class_mapping.csv"
class_mapping_df.to_csv(mapping_path, index=False)

loaded_mapping = pd.read_csv(
    mapping_path,
    dtype={"class_label": str},
)

if (
    loaded_mapping["class_label"].tolist() != class_names
    or loaded_mapping["class_index"].tolist() != list(range(N_CLASSES))
):
    raise ValueError("Saved class mapping differs from the original.")

for source, (_, metadata, primary, secondary) in target_sets.items():
    metadata_path = LABEL_DIR / f"{source}_metadata.csv"
    targets_path = LABEL_DIR / f"{source}_targets.npz"

    metadata.to_csv(metadata_path, index=False)

    sample_ids = metadata["sample_id"].to_numpy(dtype=str)

    np.savez_compressed(
        targets_path,
        sample_ids=sample_ids,
        primary_target=primary,
        secondary_target=secondary,
    )

    loaded_metadata = pd.read_csv(
        metadata_path,
        dtype={
            "sample_id": str,
            "filename": str,
            "source": str,
        },
        float_precision="round_trip",
    )

    pd.testing.assert_frame_equal(
        metadata.reset_index(drop=True),
        loaded_metadata,
        check_dtype=False,
        check_exact=True,
        obj=f"{source} metadata",
    )

    with np.load(targets_path, allow_pickle=False) as saved:
        for name, expected in [
            ("sample_ids", sample_ids),
            ("primary_target", primary),
            ("secondary_target", secondary),
        ]:
            actual = saved[name]

            if actual.dtype != expected.dtype:
                raise ValueError(
                    f"{source}: saved {name} has an unexpected dtype."
                )

            np.testing.assert_array_equal(
                actual,
                expected,
                err_msg=f"{source}: saved {name} differs from the original.",
            )

print(f"OK: label data exported and verified in {LABEL_DIR}.")

In [ ]:
# Paste this cell after Section 4, replacing the previous Sections 5 and 6.
# Dependencies: configuration and the label metadata/arrays from Sections 1-4.

from time import perf_counter
from contextlib import contextmanager
from collections import defaultdict

pipeline_started = perf_counter()
stage_seconds = defaultdict(float)
source_seconds = {}


@contextmanager
def measure_stage(stage, source):
    started = perf_counter()
    try:
        yield
    finally:
        stage_seconds[(stage, source)] += perf_counter() - started


import json
import shutil
from tqdm.auto import tqdm
from uuid import uuid4

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import tensorflow as tf
from pathlib import Path

# Set to None for the complete dataset. This limits FILES, not intervals.
PILOT_FILES_PER_SOURCE = None
NORMALIZATION_EPSILON = 1e-6
GPU_WINDOW_BATCH = 16

# Selected representation: 64 mel bands and a 20 ms hop at 32 kHz.
N_MELS = 64
HOP_LENGTH = 640
if SAMPLE_RATE != 32000 or WINDOW_SAMPLES != 160000:
    raise ValueError("This configuration expects 5-second windows at 32 kHz.")
if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("No GPU available. Enable a Kaggle GPU accelerator.")


# ============================================================
# 5 - SELECT FILES AND DEFINE SHARED PREPROCESSING
# ============================================================

def select_source_files(frame, limit, seed):
    """Select unique recordings, preserving all their annotated examples."""
    names = sorted(frame["filename"].unique())
    if limit is not None:
        if not isinstance(limit, int) or limit < 1:
            raise ValueError("File limit must be a positive integer or None.")
        if len(names) > limit:
            rng = np.random.default_rng(seed)
            names = sorted(rng.choice(names, size=limit, replace=False).tolist())
    return frame.loc[frame["filename"].isin(names)].copy()


def make_windows(waveform):
    """Preserve the waveform; right-pad only the last 5-second window."""
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim != 1 or not len(waveform) or not np.isfinite(waveform).all():
        raise ValueError("Expected a finite, nonempty mono waveform.")
    count = (len(waveform) + WINDOW_SAMPLES - 1) // WINDOW_SAMPLES
    valid = np.minimum(
        WINDOW_SAMPLES, len(waveform) - np.arange(count) * WINDOW_SAMPLES
    ).astype(np.int32)
    windows = np.pad(
        waveform, (0, count * WINDOW_SAMPLES - len(waveform))
    ).reshape(count, WINDOW_SAMPLES)
    return windows, valid


def extract_interval(waveform, start, end, sample_rate):
    """Slice in native sample coordinates, before resampling."""
    if not np.isfinite([start, end]).all():
        raise ValueError("Annotation boundaries must be finite.")
    if start < 0 or not np.isclose(
        end - start, WINDOW_SECONDS, atol=1e-6, rtol=0
    ):
        raise ValueError("Expected a nonnegative, 5-second interval.")
    left, right = int(round(start * sample_rate)), int(round(end * sample_rate))
    # Allow one sample of boundary discrepancy, as in the previous loader.
    if left >= len(waveform) or right > len(waveform) + 1:
        raise ValueError("Annotated interval exceeds the decoded audio.")
    return waveform[left:min(right, len(waveform))]


N_FRAMES = 1 + (WINDOW_SAMPLES - FRAME_LENGTH) // HOP_LENGTH
with tf.device("/GPU:0"):
    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS,
        num_spectrogram_bins=FFT_LENGTH // 2 + 1,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=MEL_LOWER_HZ,
        upper_edge_hertz=MEL_UPPER_HZ,
        dtype=tf.float32,
    )


@tf.function(input_signature=[
    tf.TensorSpec([None, WINDOW_SAMPLES], tf.float32)
])
def windows_to_log_mel(windows):
    """GPU batch processing; normalization remains independent per window."""
    with tf.device("/GPU:0"):
        spectrum = tf.signal.stft(
            windows, frame_length=FRAME_LENGTH, frame_step=HOP_LENGTH,
            fft_length=FFT_LENGTH, window_fn=tf.signal.hann_window,
            pad_end=False,
        )
        mel = tf.linalg.matmul(tf.square(tf.abs(spectrum)), mel_matrix)
        features = tf.math.log(tf.maximum(mel, LOG_EPSILON))
        mean, variance = tf.nn.moments(features, [1, 2], keepdims=True)
        features = (features - mean) / tf.sqrt(
            variance + NORMALIZATION_EPSILON
        )
        return features[..., tf.newaxis]


# Trace the function once and verify that its result is on the GPU.
probe = windows_to_log_mel(tf.zeros([1, WINDOW_SAMPLES], tf.float32))
if "GPU:0" not in probe.device.upper():
    raise RuntimeError(f"Unexpected feature output device: {probe.device}")
tf.debugging.assert_all_finite(probe, "Nonfinite GPU test features")
probe.numpy()
print(f"OK: GPU feature preprocessing initialized on {probe.device}.")
del probe


sources = {
    "audio": (audio_metadata, DATA_DIR / "train_audio",
              audio_primary_targets, audio_secondary_targets),
    "soundscape": (soundscape_metadata, DATA_DIR / "train_soundscapes",
                   soundscape_primary_targets, soundscape_secondary_targets),
}
selected_sources = {}
for source, (metadata, directory, primary, secondary) in sources.items():
    if primary.shape != (len(metadata), N_CLASSES) or secondary.shape != primary.shape:
        raise ValueError(f"{source}: target arrays do not match metadata.")
    frame = metadata.reset_index(drop=True).copy()
    frame["target_position"] = np.arange(len(frame))
    selected_sources[source] = select_source_files(frame, PILOT_FILES_PER_SOURCE, SEED)

selected_ids = pd.concat(list(selected_sources.values()))["sample_id"]
if selected_ids.empty or selected_ids.isna().any() or not selected_ids.is_unique:
    raise ValueError("Selected sample identifiers are empty, missing or duplicated.")

selection_text = "; ".join(
    f"{source}: {frame['filename'].nunique():,} files, {len(frame):,} bags"
    for source, frame in selected_sources.items()
)
print(f"OK: examples selected ({selection_text}).")


# ============================================================
# 6 - DECODE ONCE, VALIDATE FEATURES AND EXPORT
# ============================================================

mode = "full" if PILOT_FILES_PER_SOURCE is None else "pilot"
FEATURE_DIR = OUTPUT_DIR / f"spectrograms_{mode}_{uuid4().hex[:12]}"
FEATURE_DIR.mkdir(parents=True, exist_ok=False)

exported_rows = []
length_differences = []
conversion_counts = {"files": 0, "resampled": 0, "mono": 0}

setup_seconds = perf_counter() - pipeline_started
processing_started = perf_counter()

# Progress counts successfully saved bags; ETA is approximate for variable lengths.
with tqdm(
    total=len(selected_ids), desc="Spectrogram export", unit="bag", mininterval=1.0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} bags [{elapsed}<{remaining}, {rate_fmt}{postfix}]",
) as progress:
    for source, frame in selected_sources.items():
        source_started = perf_counter()
        progress.set_postfix(source=source, refresh=False)
        _, directory, primary_targets, secondary_targets = sources[source]

        for filename, examples in frame.groupby("filename", sort=True):
            path = directory / filename
            try:
                with measure_stage("Header reads", source):
                    info = sf.info(str(path))
                # Decode the file exactly once, regardless of its interval count.
                with measure_stage("Audio read + decoding", source):
                    native, original_sr = sf.read(
                        str(path), dtype="float32", always_2d=True
                    )
                with measure_stage("Decoded audio checks", source):
                    if original_sr <= 0 or not len(native) or not np.isfinite(native).all():
                        raise ValueError("Empty or nonfinite decoded audio.")

                    if len(native) != info.frames:
                        length_differences.append({
                            "source": source, "filename": filename,
                            "header_samples": info.frames, "decoded_samples": len(native),
                            "difference_seconds": (info.frames - len(native)) / original_sr,
                        })

                with measure_stage("Mono conversion", source):
                    needs_mono = native.shape[1] != 1
                    mono_native = native.mean(axis=1) if needs_mono else native[:, 0]
                    needs_resampling = original_sr != SAMPLE_RATE

                for row in examples.itertuples(index=False):
                    with measure_stage("Interval + resampling + windowing", source):
                        # Soundscape labels apply only to their annotated interval.
                        segment = mono_native if source == "audio" else extract_interval(
                            mono_native, float(row.start), float(row.end), original_sr
                        )
                        if needs_resampling:
                            segment = librosa.resample(
                                segment, orig_sr=original_sr, target_sr=SAMPLE_RATE
                            )
                        segment = np.asarray(segment, dtype=np.float32)

                        if source == "soundscape":
                            if abs(len(segment) - WINDOW_SAMPLES) > 1:
                                raise ValueError(f"Unexpected interval length: {row.sample_id}")
                            # A rounding difference must not create a second instance.
                            segment = segment[:WINDOW_SAMPLES]

                        windows, valid_samples = make_windows(segment)
                        if source == "soundscape" and len(windows) != 1:
                            raise ValueError(f"Soundscape bag is not a singleton: {row.sample_id}")

                    with measure_stage("GPU features + transfers", source):
                        features = np.empty(
                            (len(windows), N_FRAMES, N_MELS, 1), dtype=np.float32
                        )
                        for start in range(0, len(windows), GPU_WINDOW_BATCH):
                            stop = min(start + GPU_WINDOW_BATCH, len(windows))
                            features[start:stop] = windows_to_log_mel(
                                tf.convert_to_tensor(windows[start:stop], dtype=tf.float32)
                            ).numpy()
                    with measure_stage("Feature + target checks", source):
                        if not np.isfinite(features).all():
                            raise ValueError(f"Nonfinite features: {row.sample_id}")

                        position = int(row.target_position)
                        primary = primary_targets[position]
                        secondary = secondary_targets[position]
                        for name, target in [("primary", primary), ("secondary", secondary)]:
                            if target.shape != (N_CLASSES,) or not np.isin(target, [0, 1]).all():
                                raise ValueError(f"Invalid {name} target: {row.sample_id}")
                        if source == "soundscape" and secondary.any():
                            raise ValueError(f"Soundscape has secondary targets: {row.sample_id}")

                    recording_id = f"{source}:{filename}"
                    payload = {
                        "features": features, "valid_samples": valid_samples,
                        "primary_target": primary, "secondary_target": secondary,
                        "sample_id": np.asarray(row.sample_id),
                        "recording_id": np.asarray(recording_id),
                        "source": np.asarray(source),
                        "rating": np.asarray(row.rating, dtype=np.float64),
                    }
                    output_path = FEATURE_DIR / f"bag_{len(exported_rows):06d}.npz"
                    with measure_stage("Compression + file write", source):
                        np.savez_compressed(output_path, **payload)

                    # Export only: do not reopen or decompress saved bags.

                    exported_rows.append({
                        "sample_id": row.sample_id, "recording_id": recording_id,
                        "source": source, "filename": filename,
                        "rating": float(row.rating),
                        "start": float(row.start) if source == "soundscape" else np.nan,
                        "end": float(row.end) if source == "soundscape" else np.nan,
                        "feature_file": output_path.name, "n_windows": len(windows),
                        "n_valid_samples": int(valid_samples.sum()),
                        "saved_bytes": output_path.stat().st_size,
                    })
                    progress.update(1)
                    del payload, features, windows, segment

                conversion_counts["files"] += 1
                conversion_counts["resampled"] += int(needs_resampling)
                conversion_counts["mono"] += int(needs_mono)
                del native, mono_native

            except Exception as error:
                raise RuntimeError(f"Preparation failed for {source}:{filename}: {error}") from error

        source_seconds[source] = perf_counter() - source_started

processing_seconds = perf_counter() - processing_started

feature_manifest = pd.DataFrame(exported_rows)
if len(feature_manifest) != len(selected_ids) or set(feature_manifest.sample_id) != set(selected_ids):
    raise ValueError("Exported examples differ from the selected examples.")
feature_manifest.to_csv(FEATURE_DIR / "manifest.csv", index=False)

config = {
    "sample_rate": SAMPLE_RATE, "window_seconds": WINDOW_SECONDS,
    "window_samples": WINDOW_SAMPLES, "frame_length": FRAME_LENGTH,
    "hop_length": HOP_LENGTH, "fft_length": FFT_LENGTH, "n_mels": N_MELS,
    "mel_lower_hz": MEL_LOWER_HZ, "mel_upper_hz": MEL_UPPER_HZ,
    "log_epsilon": LOG_EPSILON, "normalization": "per_window_standardization",
    "normalization_epsilon": NORMALIZATION_EPSILON, "stft_pad_end": False,
    "stft_window": "periodic_hann", "feature_dtype": "float32",
    "shape_per_window": [N_FRAMES, N_MELS, 1], "seed": SEED,
    "pilot_files_per_source": PILOT_FILES_PER_SOURCE,
    "tensorflow_version": tf.__version__,
    "feature_compute_device": "/GPU:0",
    "gpu_window_batch": GPU_WINDOW_BATCH,
    "verify_saved_content": False,
    "rating_semantics": "Original audio rating preserved, including zero; soundscape rating is NaN. Not used as a model input or weight by default.",
    "target_semantics": {
        "audio": "primary species and secondary species in separate vectors; bag-level labels",
        "soundscape": "all interval species in primary vector; secondary vector is zero",
    },
}
for name, value in [("config.json", config), ("class_names.json", class_names)]:
    (FEATURE_DIR / name).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

pd.DataFrame(length_differences, columns=[
    "source", "filename", "header_samples", "decoded_samples", "difference_seconds"
]).to_csv(FEATURE_DIR / "audio_length_differences.csv", index=False)

total_files = conversion_counts["files"]
if conversion_counts["resampled"]:
    print(f"OK: {conversion_counts['resampled']:,} of {total_files:,} files required resampling to {SAMPLE_RATE:,} Hz.")
else:
    print(f"OK: all {total_files:,} processed files were already sampled at {SAMPLE_RATE:,} Hz.")
if conversion_counts["mono"]:
    print(f"OK: {conversion_counts['mono']:,} of {total_files:,} files required conversion to mono.")
else:
    print(f"OK: all {total_files:,} processed files were already mono.")
if length_differences:
    print(f"NOTE: {len(length_differences):,} header/decoded length differences recorded in audio_length_differences.csv; inspect this report.")

summary = feature_manifest.groupby("source").agg(
    recordings=("recording_id", "nunique"), bags=("sample_id", "size"),
    windows=("n_windows", "sum"), saved_bytes=("saved_bytes", "sum"),
)
summary["saved_GiB"] = summary.pop("saved_bytes") / 1024**3
display(summary.round(3))
print(f"OK: all {len(feature_manifest):,} spectrogram bags exported (no read-back verification) to {FEATURE_DIR}.")
print(f"Remaining free disk space: {shutil.disk_usage(FEATURE_DIR).free / 1024**3:.2f} GiB")


# ============================================================
# TIMING SUMMARY - PRINTED ONLY AFTER THE PROCESSING HAS FINISHED
# ============================================================

elapsed_seconds = perf_counter() - pipeline_started
stages = [
    "Header reads",
    "Audio read + decoding",
    "Decoded audio checks",
    "Mono conversion",
    "Interval + resampling + windowing",
    "GPU features + transfers",
    "Feature + target checks",
    "Compression + file write",
]

# Attribute time outside the explicit stage blocks to bookkeeping/other.
# Stage values do not overlap and include host waiting and GPU transfers.
for source, wall_seconds in source_seconds.items():
    measured_seconds = sum(
        seconds for (stage, src), seconds in stage_seconds.items()
        if src == source
    )
    stage_seconds[("Bookkeeping / other", source)] = max(
        0.0, wall_seconds - measured_seconds
    )
stages.append("Bookkeeping / other")

timing_table = pd.DataFrame({
    source: [stage_seconds[(stage, source)] for stage in stages]
    for source in source_seconds
}, index=stages)
timing_table.index.name = "stage"
timing_table["total_seconds"] = timing_table.sum(axis=1)
timing_table["processing_percent"] = (
    timing_table["total_seconds"] / processing_seconds * 100
    if processing_seconds > 0 else 0.0
)
timing_table.to_csv(FEATURE_DIR / "timing_by_stage.csv")

runtime_summary = {
    "setup_and_gpu_warmup_seconds": setup_seconds,
    "processing_seconds": processing_seconds,
    "finalization_seconds": max(0.0, elapsed_seconds - setup_seconds - processing_seconds),
    "total_elapsed_seconds": elapsed_seconds,
    "readback_verification": False,
    "notes": "Audio read includes file I/O and decoding. GPU time includes tensor conversion, computation, synchronization and copying results to CPU. Compression/write is combined. Times are wall-clock durations, not pure device kernel times.",
}
(FEATURE_DIR / "timing_summary.json").write_text(
    json.dumps(runtime_summary, indent=2), encoding="utf-8"
)

print("\nStage timings in seconds; percentages refer to the processing loop:")
display(timing_table.round(3))
print(f"Setup and GPU warm-up: {setup_seconds:.2f} s")
print(f"Processing: {processing_seconds:.2f} s")
print(f"Finalization: {runtime_summary['finalization_seconds']:.2f} s")
print(f"Total elapsed: {elapsed_seconds:.2f} s ({elapsed_seconds / 60:.2f} min)")
print("OK: timing reports saved alongside the spectrograms.")

## 7. Recording-level train/validation/test split
Split the exported bags without moving or duplicating NPZ files. Audio ratios
are approximately 80/10/10 with primary-class stratification. Classes with only
one or two audio recordings stay in training. Soundscape ratios are 70/15/15
by source recording; all annotated intervals from one recording remain together.
A greedy selection ensures that classes absent from audio training are supplied
by soundscape training. Validation/test may lack rare classes; coverage is reported.
This grouping does not detect copies under different filenames or shared site/day.
Ratings are preserved as metadata and are not used to select the split.

In [ ]:
"""Recording-level splits for an already exported MIL spectrogram dataset."""
import json
from collections import Counter
import numpy as np
import pandas as pd

SPLIT_NAMES = ["train", "val", "test"]


def _validate_ratios(ratios):
    values = np.asarray(ratios, dtype=float)
    if values.shape != (3,) or not np.isfinite(values).all() or (values <= 0).any() or not np.isclose(values.sum(), 1):
        raise ValueError("Split ratios must contain three positive values summing to one.")


def split_prepared_dataset(frame, classes, seed, audio_ratios, soundscape_ratios,
                           require_full_coverage=True):
    """Assign recordings, never individual windows, to a split.

    Audio is stratified by its primary species. Classes with fewer than three
    recordings remain in training. A greedy soundscape selection supplies any
    classes still missing from training; validation/test are not used for fitting.
    Ratios for soundscapes refer to recordings, not annotated intervals.
    """
    _validate_ratios(audio_ratios)
    _validate_ratios(soundscape_ratios)
    required_columns = {"sample_id", "recording_id", "source", "primary_labels", "secondary_labels"}
    if required_columns - set(frame.columns):
        raise ValueError(f"Missing split fields: {sorted(required_columns - set(frame.columns))}")
    result = frame.sort_values("sample_id").reset_index(drop=True).copy()
    if result.empty or result["sample_id"].isna().any() or not result["sample_id"].is_unique:
        raise ValueError("Example identifiers must be nonempty and unique.")
    if not result["source"].isin(["audio", "soundscape"]).all():
        raise ValueError("Unexpected dataset source.")
    if result["recording_id"].isna().any():
        raise ValueError("Recording identifiers must be present.")
    if result.groupby("recording_id")["source"].nunique().gt(1).any():
        raise ValueError("A recording identifier is shared by different sources.")
    result["all_labels"] = [
        sorted(set(primary) | set(secondary))
        for primary, secondary in zip(result["primary_labels"], result["secondary_labels"])
    ]
    represented = {v for labels in result["all_labels"] for v in labels}
    unknown = represented - set(classes)
    if unknown:
        raise ValueError(f"Unknown classes: {sorted(unknown)}")
    unavailable = set(classes) - represented
    if unavailable and require_full_coverage:
        raise ValueError(f"Classes absent from the exported dataset: {sorted(unavailable)}")
    # A pilot can cover only classes actually present in the selection.
    desired_coverage = set(classes) if require_full_coverage else represented
    result["split"] = ""
    rng = np.random.default_rng(seed)

    audio = result[result["source"] == "audio"].copy()
    if audio["recording_id"].duplicated().any():
        raise ValueError("Audio must have one full-recording bag per recording.")
    if not audio["primary_labels"].map(len).eq(1).all():
        raise ValueError("Every audio bag must have one primary species.")
    audio["primary_class"] = audio["primary_labels"].map(lambda labels: labels[0])
    for _, group in audio.groupby("primary_class", sort=True):
        indices = group.index.to_numpy()[rng.permutation(len(group))]
        count = len(indices)
        n_val = n_test = 0
        if count >= 3:
            n_val = max(1, round(count * audio_ratios[1]))
            n_test = max(1, round(count * audio_ratios[2]))
            while n_val + n_test > count - 1:
                if n_val >= n_test and n_val > 1:
                    n_val -= 1
                else:
                    n_test -= 1
        n_train = count - n_val - n_test
        result.loc[indices[:n_train], "split"] = "train"
        result.loc[indices[n_train:n_train + n_val], "split"] = "val"
        result.loc[indices[n_train + n_val:], "split"] = "test"

    covered = {v for labels in result.loc[result["split"] == "train", "all_labels"] for v in labels}
    needed = desired_coverage - covered
    soundscapes = result[result["source"] == "soundscape"]
    recording_labels = {
        recording: {v for labels in group["all_labels"] for v in labels}
        for recording, group in soundscapes.groupby("recording_id", sort=True)
    }
    remaining = sorted(recording_labels)
    mandatory = []
    while needed:
        gains = {name: len(recording_labels[name] & needed) for name in remaining}
        if not gains or max(gains.values()) == 0:
            raise ValueError(f"Cannot provide training examples for: {sorted(needed)}")
        gain = max(gains.values())
        candidates = sorted(name for name, value in gains.items() if value == gain)
        chosen = str(rng.choice(candidates))
        mandatory.append(chosen)
        remaining.remove(chosen)
        needed -= recording_labels[chosen]

    n_total = len(recording_labels)
    n_train = round(n_total * soundscape_ratios[0])
    n_val = round(n_total * soundscape_ratios[1])
    if n_total >= 3:
        n_train = min(n_total - 2, max(1, n_train))
        n_val = min(n_total - n_train - 1, max(1, n_val))
    if len(mandatory) > n_train:
        raise ValueError("Training coverage requires more soundscape recordings than the configured train capacity.")
    rng.shuffle(remaining)
    train_recordings = set(mandatory) | set(remaining[:n_train - len(mandatory)])
    remaining = [name for name in remaining if name not in train_recordings]
    val_recordings = set(remaining[:n_val])
    assignment = {
        name: "train" if name in train_recordings else "val" if name in val_recordings else "test"
        for name in recording_labels
    }
    result.loc[soundscapes.index, "split"] = soundscapes["recording_id"].map(assignment)

    if not result["split"].isin(SPLIT_NAMES).all():
        raise ValueError("Some examples have no valid split assignment.")
    if result.groupby("recording_id")["split"].nunique().gt(1).any():
        raise ValueError("A recording appears in multiple splits.")
    if len(result) != len(frame) or set(result["sample_id"]) != set(frame["sample_id"]):
        raise ValueError("Examples were lost or duplicated during splitting.")
    if set(result["split"]) != set(SPLIT_NAMES):
        raise ValueError("At least one split is empty; more recordings are needed.")
    coverage = {
        split: {v for labels in result.loc[result["split"] == split, "all_labels"] for v in labels}
        for split in SPLIT_NAMES
    }
    if desired_coverage - coverage["train"]:
        raise ValueError("Training class coverage validation failed.")
    counts = {
        split: Counter(v for labels in result.loc[result["split"] == split, "all_labels"] for v in labels)
        for split in SPLIT_NAMES
    }
    class_coverage = pd.DataFrame({
        "class_label": classes,
        **{f"{split}_positive_bags": [counts[split][label] for label in classes] for split in SPLIT_NAMES},
    })
    return result, class_coverage


def export_dataset_splits(frame, class_coverage, output_dir, seed, audio_ratios,
                          soundscape_ratios, full_export):
    """Write only small manifests; keep all spectrogram bags in place."""
    from pathlib import Path
    output_dir = Path(output_dir)
    split_dir = output_dir / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    exported = frame.drop(columns=["all_labels"], errors="ignore").copy()
    for column in ["primary_labels", "secondary_labels"]:
        exported[column] = exported[column].map(lambda labels: json.dumps(labels, ensure_ascii=False))
    exported.to_csv(output_dir / "manifest.csv", index=False)
    for split in SPLIT_NAMES:
        exported.loc[exported["split"] == split].to_csv(split_dir / f"{split}.csv", index=False)
    class_coverage.to_csv(split_dir / "class_coverage.csv", index=False)
    settings = {
        "seed": seed, "audio_ratios": list(audio_ratios),
        "soundscape_recording_ratios": list(soundscape_ratios),
        "unit": "recording_id", "full_export": bool(full_export),
        "require_full_training_coverage": bool(full_export),
        "audio_policy": "primary-class stratification; fewer than 3 recordings stay in train",
        "soundscape_policy": "greedy training class coverage followed by random recording split",
        "limitations": "Recording identifiers do not detect audio copies or shared site/day between distinct recordings.",
    }
    (split_dir / "split_config.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    return exported


# Attach label descriptions by stable sample identifiers, never by split row order.
label_rows = []
for source, frame, metadata in [
    ("audio", train_clean_df, audio_metadata),
    ("soundscape", soundscape_clean_df, soundscape_metadata),
]:
    labels = metadata[["sample_id"]].reset_index(drop=True).copy()
    labels["primary_labels"] = frame["primary_list"].tolist()
    labels["secondary_labels"] = frame["secondary_list"].tolist()
    label_rows.append(labels)
label_table = pd.concat(label_rows, ignore_index=True)
split_input = feature_manifest.merge(
    label_table, on="sample_id", how="left", validate="one_to_one", indicator=True
)
if not split_input["_merge"].eq("both").all():
    raise ValueError("Some exported bags have no matching labels.")
split_input = split_input.drop(columns="_merge")

audio_ratios = [AUDIO_TRAIN_RATIO, AUDIO_VAL_RATIO, AUDIO_TEST_RATIO]
soundscape_ratios = [SOUNDSCAPE_TRAIN_RATIO, SOUNDSCAPE_VAL_RATIO, SOUNDSCAPE_TEST_RATIO]
full_export = PILOT_FILES_PER_SOURCE is None
split_frame, class_coverage = split_prepared_dataset(
    split_input, class_names, SEED, audio_ratios, soundscape_ratios,
    require_full_coverage=full_export,
)
feature_manifest = export_dataset_splits(
    split_frame, class_coverage, FEATURE_DIR, SEED,
    audio_ratios, soundscape_ratios, full_export,
)
split_summary = split_frame.groupby(["source", "split"]).agg(
    recordings=("recording_id", "nunique"), bags=("sample_id", "size"),
    windows=("n_windows", "sum"), available_ratings=("rating", "count"),
).reindex(pd.MultiIndex.from_product(
    [["audio", "soundscape"], SPLIT_NAMES], names=["source", "split"]
), fill_value=0)
split_summary.to_csv(FEATURE_DIR / "splits" / "summary.csv")
display(split_summary)
display(pd.DataFrame({
    "split": SPLIT_NAMES,
    "represented_classes": [int(class_coverage[f"{s}_positive_bags"].gt(0).sum()) for s in SPLIT_NAMES],
    "taxonomy_classes": N_CLASSES,
}))
if not full_export:
    print("NOTE: pilot split only; full taxonomy coverage is not required for a pilot.")
print("OK: recording-level splits, example completeness and training coverage verified; split manifests saved.")

## 8. Loader for the future neural network
Each NPZ stores `features [instances, 247, 64, 1]`, `primary_target [234]`,
`secondary_target [234]`, valid waveform sample counts, identifiers and `rating`.
Audio labels describe the complete bag; soundscape labels describe one annotated
interval. For soundscapes, all species are in primary_target and secondary_target
is zero. An unknown rating is NaN, not zero. Source audio ratings equal to zero
are preserved as zero; no quality filtering or loss weighting is applied.

The loader yields `(inputs, targets)`. Inputs contain features and a boolean
instance mask. Targets contain the two separate label vectors. Padding is added
only within the current batch; the partially padded last real audio window stays
valid in the mask. Rating and identifiers remain accessible through split metadata.
Dataset construction is lazy: no automatic spectrogram read-back is performed.
The training notebook can combine or weight the two targets explicitly.

In [ ]:
"""Read exported MIL bags without decoding OGG or computing spectrograms."""
from pathlib import Path
import json
import numpy as np
import pandas as pd


def load_split_metadata(dataset_dir, split):
    """Return example identifiers, relative feature paths and optional ratings."""
    if split not in {"train", "val", "test"}:
        raise ValueError("Split must be train, val or test.")
    return pd.read_csv(
        Path(dataset_dir) / "splits" / f"{split}.csv",
        dtype={"sample_id": str, "recording_id": str, "source": str, "filename": str},
    )


def make_prepared_dataset(dataset_dir, split, batch_size=1, shuffle=None, seed=42,
                          sample_ids=None):
    """Load only saved NPZ files and pad bag lengths within each batch.

    Returns (inputs, targets): inputs contain features and a boolean mask;
    targets contain primary_target and secondary_target. Rating remains in
    load_split_metadata() and in each NPZ; it is not automatically a model
    feature or a loss weight. Missing soundscape ratings are represented by NaN.
    Creating this dataset does not read any spectrogram until it is iterated.
    """
    import tensorflow as tf
    directory = Path(dataset_dir)
    metadata = load_split_metadata(directory, split)
    # A bounded selection is useful for the final read-back demonstration.
    if sample_ids is not None:
        requested = list(sample_ids)
        if len(requested) != len(set(requested)):
            raise ValueError("Requested sample identifiers must be unique.")
        missing = set(requested) - set(metadata["sample_id"])
        if missing:
            raise ValueError(f"Samples not present in this split: {sorted(missing)}")
        metadata = metadata.set_index("sample_id").loc[requested].reset_index()
    config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    classes = json.loads((directory / "class_names.json").read_text(encoding="utf-8"))
    shape = tuple(config["shape_per_window"])
    n_classes = len(classes)
    if metadata.empty or batch_size < 1:
        raise ValueError("A nonempty split and positive batch size are required.")
    if shuffle is None:
        shuffle = split == "train"
    def read_example(index):
        row = metadata.iloc[int(index)]
        path = directory / row["feature_file"]
        if path.resolve().parent != directory.resolve():
            raise ValueError(f"Unexpected feature path: {path}")
        with np.load(path, allow_pickle=False) as saved:
            if saved["sample_id"].item() != row["sample_id"]:
                raise ValueError("Saved example does not match the split manifest.")
            features = saved["features"]
            primary = saved["primary_target"].astype(np.float32)
            secondary = saved["secondary_target"].astype(np.float32)
        if features.shape[1:] != shape or features.shape[0] < 1:
            raise ValueError("Unexpected spectrogram bag shape.")
        return features, np.ones(len(features), dtype=bool), primary, secondary
    def load_index(index):
        features, mask, primary, secondary = tf.numpy_function(
            read_example, [index], [tf.float32, tf.bool, tf.float32, tf.float32]
        )
        features.set_shape([None, *shape])
        mask.set_shape([None])
        primary.set_shape([n_classes])
        secondary.set_shape([n_classes])
        return {"features": features, "mask": mask}, {
            "primary_target": primary, "secondary_target": secondary,
        }
    ds = tf.data.Dataset.range(len(metadata))
    if shuffle:
        ds = ds.shuffle(len(metadata), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(load_index, num_parallel_calls=1, deterministic=True)
    ds = ds.padded_batch(
        batch_size,
        padded_shapes=(
            {"features": [None, *shape], "mask": [None]},
            {"primary_target": [n_classes], "secondary_target": [n_classes]},
        ),
        padding_values=(
            {"features": np.float32(0), "mask": False},
            {"primary_target": np.float32(0), "secondary_target": np.float32(0)},
        ),
    )
    return ds.prefetch(1)


train_metadata = load_split_metadata(FEATURE_DIR, "train")
val_metadata = load_split_metadata(FEATURE_DIR, "val")
test_metadata = load_split_metadata(FEATURE_DIR, "test")

train_ds = make_prepared_dataset(FEATURE_DIR, "train", batch_size=BATCH_SIZE, seed=SEED)
val_ds = make_prepared_dataset(FEATURE_DIR, "val", batch_size=BATCH_SIZE, seed=SEED)
test_ds = make_prepared_dataset(FEATURE_DIR, "test", batch_size=BATCH_SIZE, seed=SEED)

# Intentionally do not iterate the datasets here: saved bags are read only when used.
print("OK: train, validation and test loaders created without recomputing spectrograms.")
print("Final dataset directory:", FEATURE_DIR)

## 9. Final dataset handoff check — no training
Read back at most one audio and one soundscape bag per split (up to six distinct
examples). Prefer a small multi-window audio bag to exercise variable bag lengths.
Compare saved shapes, identifiers, label vectors, rating and real sample counts
with the split manifests. Then use the actual loader on only two selected train
examples to check padding, masks and labels. The two train examples are read once
more by that loader. No full-dataset read-back, model or optimization is performed.
These checks verify the sampled examples; they do not certify every NPZ file.

In [ ]:
"""Small read-back check for Milestone I; no neural network or training."""
import json
from pathlib import Path
import numpy as np
import pandas as pd


def inspect_saved_example(dataset_dir, row, config, classes):
    """Read one saved bag and compare it with its exported metadata."""
    directory = Path(dataset_dir).resolve()
    path = directory / row["feature_file"]
    if path.resolve().parent != directory:
        raise ValueError(f"Unexpected feature path: {path}")
    with np.load(path, allow_pickle=False) as saved:
        needed = {"features", "primary_target", "secondary_target", "valid_samples",
                  "sample_id", "recording_id", "source", "rating"}
        if needed - set(saved.files):
            raise ValueError(f"Missing saved fields: {sorted(needed - set(saved.files))}")
        # Each NPZ member is read exactly once.
        data = {name: saved[name] for name in needed}

    for name in ["sample_id", "recording_id", "source"]:
        if data[name].item() != row[name]:
            raise ValueError(f"Saved {name} does not match manifest: {row['sample_id']}")
    shape = (int(row["n_windows"]), *config["shape_per_window"])
    features = data["features"]
    if features.shape != shape or features.dtype != np.float32 or not np.isfinite(features).all():
        raise ValueError(f"Invalid saved features: {row['sample_id']}")
    n_classes = len(classes)
    for name, label_column in [("primary_target", "primary_labels"),
                                ("secondary_target", "secondary_labels")]:
        target = data[name]
        if target.shape != (n_classes,) or target.dtype != np.uint8 or not np.isin(target, [0, 1]).all():
            raise ValueError(f"Invalid saved {name}: {row['sample_id']}")
        actual_labels = {classes[i] for i in np.flatnonzero(target)}
        expected_labels = set(json.loads(row[label_column]))
        if actual_labels != expected_labels:
            raise ValueError(f"Saved labels differ from manifest: {row['sample_id']}")
    valid = data["valid_samples"]
    if valid.shape != (len(features),) or not np.issubdtype(valid.dtype, np.integer):
        raise ValueError("Invalid valid_samples shape or dtype.")
    if np.any(valid <= 0) or np.any(valid > config["window_samples"]):
        raise ValueError("Invalid count of real waveform samples.")
    if int(valid.sum()) != int(row["n_valid_samples"]):
        raise ValueError("Real waveform sample count differs from manifest.")
    if len(valid) > 1 and not np.all(valid[:-1] == config["window_samples"]):
        raise ValueError("Only the final waveform window may be partially padded.")

    rating_array = data["rating"]
    if rating_array.shape != () or not np.issubdtype(rating_array.dtype, np.floating):
        raise ValueError("Rating must be a floating-point scalar.")
    actual_rating = float(rating_array)
    expected_rating = float(row["rating"])
    if not np.isclose(actual_rating, expected_rating, equal_nan=True, atol=1e-12, rtol=0):
        raise ValueError("Saved rating differs from manifest.")
    if row["source"] == "soundscape":
        if len(features) != 1 or data["secondary_target"].any() or not np.isnan(actual_rating):
            raise ValueError("Invalid soundscape singleton, secondary labels or rating.")
    elif row["source"] == "audio":
        if data["primary_target"].sum() != 1:
            raise ValueError("An audio recording must have exactly one primary class.")
    else:
        raise ValueError("Unexpected data source.")
    return data


def choose_readback_examples(metadata):
    """Choose at most one modest-size example per source in this split.

    Prefer a multi-window audio bag when available so that pairing it with a
    soundscape singleton exercises real variable-length padding.
    """
    rows = []
    for source in ["audio", "soundscape"]:
        candidates = metadata[metadata["source"] == source]
        if candidates.empty:
            continue
        if source == "audio" and candidates["n_windows"].gt(1).any():
            candidates = candidates[candidates["n_windows"].gt(1)]
        rows.append(candidates.sort_values(["n_windows", "sample_id"]).iloc[0])
    return rows


def summarize_saved_splits(split_metadata, classes):
    """Summarize manifests only; do not read all feature files."""
    rows = []
    for split in ["train", "val", "test"]:
        frame = split_metadata[split]
        represented = set()
        for primary, secondary in zip(frame["primary_labels"], frame["secondary_labels"]):
            represented.update(json.loads(primary))
            represented.update(json.loads(secondary))
        audio = frame[frame["source"] == "audio"]
        soundscape = frame[frame["source"] == "soundscape"]
        rows.append({
            "split": split,
            "audio_recordings": audio["recording_id"].nunique(),
            "soundscape_recordings": soundscape["recording_id"].nunique(),
            "audio_bags": len(audio), "soundscape_bags": len(soundscape),
            "total_windows": int(frame["n_windows"].sum()),
            "represented_classes": len(represented), "taxonomy_classes": len(classes),
        })
    combined = pd.concat(list(split_metadata.values()), ignore_index=True)
    if not combined["sample_id"].is_unique:
        raise ValueError("An example is duplicated across split manifests.")
    if combined.groupby("recording_id")["split"].nunique().gt(1).any():
        raise ValueError("A recording is shared across split manifests.")
    return pd.DataFrame(rows)


# Load small metadata files and the representation contract.
dataset_dir = Path(FEATURE_DIR)
saved_config = json.loads((dataset_dir / "config.json").read_text(encoding="utf-8"))
saved_classes = json.loads((dataset_dir / "class_names.json").read_text(encoding="utf-8"))
split_metadata = {split: load_split_metadata(dataset_dir, split) for split in ["train", "val", "test"]}
dataset_summary = summarize_saved_splits(split_metadata, saved_classes)
all_metadata = pd.concat(list(split_metadata.values()), ignore_index=True)
master_manifest = pd.read_csv(dataset_dir / "manifest.csv", dtype={"sample_id": str})
if len(master_manifest) != len(all_metadata) or set(master_manifest["sample_id"]) != set(all_metadata["sample_id"]):
    raise ValueError("Split manifests do not cover the master manifest exactly.")

# Read a bounded number of real saved examples; no OGG processing occurs here.
sample_reports = []
checked_examples = {}
selected_by_split = {}
for split, metadata in split_metadata.items():
    selected_by_split[split] = choose_readback_examples(metadata)
    for row in selected_by_split[split]:
        data = inspect_saved_example(dataset_dir, row, saved_config, saved_classes)
        checked_examples[row["sample_id"]] = data
        sample_reports.append({
            "split": split, "source": row["source"], "sample_id": row["sample_id"],
            "feature_shape": str(tuple(data["features"].shape)),
            "primary_species": int(data["primary_target"].sum()),
            "secondary_species": int(data["secondary_target"].sum()),
            "rating": float(data["rating"]), "status": "OK",
        })
print(f"OK: {len(sample_reports)} saved examples read back; identifiers, shapes, labels and ratings verified.")

# Limit this dataset explicitly so prefetch cannot read unrelated training bags.
probe_rows = selected_by_split["train"]
if not probe_rows:
    raise ValueError("No training examples are available for a loader check.")
probe_ids = [row["sample_id"] for row in probe_rows]
probe_ds = make_prepared_dataset(
    dataset_dir, "train", batch_size=len(probe_ids), shuffle=False,
    sample_ids=probe_ids,
)
probe_inputs, probe_targets = next(iter(probe_ds))
batch_features = probe_inputs["features"].numpy()
batch_mask = probe_inputs["mask"].numpy()
expected_counts = [len(checked_examples[sample_id]["features"]) for sample_id in probe_ids]
expected_shape = (len(probe_ids), max(expected_counts), *saved_config["shape_per_window"])
if batch_features.shape != expected_shape or batch_mask.dtype != np.bool_:
    raise ValueError("Unexpected loader batch shape or mask dtype.")
for index, sample_id in enumerate(probe_ids):
    original = checked_examples[sample_id]
    count = len(original["features"])
    expected_mask = np.arange(max(expected_counts)) < count
    np.testing.assert_array_equal(batch_mask[index], expected_mask)
    np.testing.assert_array_equal(batch_features[index, :count], original["features"])
    if np.any(batch_features[index, count:] != 0):
        raise ValueError("Batch padding contains nonzero feature values.")
    for target in ["primary_target", "secondary_target"]:
        np.testing.assert_array_equal(
            probe_targets[target].numpy()[index], original[target].astype(np.float32)
        )
padding_exercised = len(set(expected_counts)) > 1
print("OK: saved-data loader preserves inputs and separate targets; batch padding and masks verified.")
if not padding_exercised:
    print("NOTE: selected examples have equal bag lengths; no extra batch padding was needed.")

sample_table = pd.DataFrame(sample_reports)
sample_table.to_csv(dataset_dir / "example_readback_checks.csv", index=False)
dataset_summary.to_csv(dataset_dir / "dataset_summary.csv", index=False)
validation_report = {
    "status": "passed", "model_training_performed": False,
    "scope": "manifest integrity, sampled NPZ read-back, and one bounded loader batch",
    "all_feature_files_read": False,
    "distinct_examples_read_back": len(sample_reports),
    "loader_batch_examples": len(probe_ids),
    "variable_length_padding_exercised": padding_exercised,
    "batch_shape": list(batch_features.shape),
    "total_bags": len(all_metadata),
    "split_bags": {split: len(frame) for split, frame in split_metadata.items()},
    "full_export": saved_config["pilot_files_per_source"] is None,
}
(dataset_dir / "dataset_validation.json").write_text(
    json.dumps(validation_report, indent=2), encoding="utf-8"
)

print("\nFinal dataset summary:")
display(dataset_summary)
print("\nSampled saved examples:")
display(sample_table)
print("OK: Milestone I data handoff checks completed. No model was trained.")
print("Preserve this complete directory:", dataset_dir)

# Release the small demonstration arrays; keep the lazy split loaders available.
del checked_examples, data, original, probe_ds, probe_inputs, probe_targets
del batch_features, batch_mask

## 10. How to use the prepared dataset

### What has been produced

The complete export directory is the dataset. `manifest.csv` lists all examples;
`splits/train.csv`, `val.csv` and `test.csv` select examples without copying the
spectrogram files. `class_names.json` fixes the shared label order and
`config.json` records the feature representation. Each NPZ stores a variable-size
spectrogram bag, two label vectors, valid waveform sample counts, identifiers
and rating. Relative feature paths permit moving the directory as one unit.

An audio recording keeps its whole sequence of 5-second windows with bag-level
labels. An annotated soundscape interval is a singleton bag. Unannotated intervals
are excluded. Primary and secondary audio labels remain separate. All annotated
soundscape species occupy primary_target, with a zero secondary_target. Ratings
are metadata: zero is preserved, while unavailable soundscape ratings are NaN.
Rating is not automatically used as an input or a training weight.

The final checks write `dataset_summary.csv`, `example_readback_checks.csv` and
`dataset_validation.json`. They inspect only a few feature files and one real
loader batch; they do not claim every saved feature file was re-read. The
`full_export` field distinguishes a complete export from a pilot.

### Preserve the files

Save the Kaggle version including file outputs, download the export directory,
or preserve it as a separate Kaggle dataset. Keeping the notebook source alone
does not preserve working-session files. Do not execute the old cleanup cells
after producing the export: they delete pilot and full spectrogram directories.
If uploading to GitHub, commit the notebook and documentation and reference the
large dataset separately, subject to the source data license/competition terms.

### Load in a later notebook

Attach the saved output/dataset. Copy the functions from Section 8, or import
`load_prepared_dataset.py` if that helper is available. Set the dataset directory
to the mounted folder that directly contains `config.json` and `manifest.csv`:

```python
dataset_dir = Path("/kaggle/input/YOUR-SAVED-DATASET/EXPORT-FOLDER")
train_ds = make_prepared_dataset(dataset_dir, "train", batch_size=1)
val_ds = make_prepared_dataset(dataset_dir, "val", batch_size=1)
test_ds = make_prepared_dataset(dataset_dir, "test", batch_size=1)
train_metadata = load_split_metadata(dataset_dir, "train")
```

For the current representation, inputs are `features [batch, max_instances, 247,
64, 1]` and `mask [batch, max_instances]`; targets are `primary_target [batch, 234]`
and `secondary_target [batch, 234]`. The mask excludes only artificial batch
instances; a partially waveform-padded final real window remains valid.
The loader reads NPZ features and never recomputes OGG spectrograms. Original OGG
input is not required to use this export. Batch size 1 is a conservative default.

Training, model architecture, loss weighting, optimization and predictive metrics
belong to Milestone II and are intentionally absent. For Milestone I, publish
this notebook and a README describing the environment, source dataset, parameters,
execution order and location of the prepared train/validation/test data.